# Mecanum Platform Dynamics Simulator (Julia) — Multicomponent LuGre Friction & ASMC with Yaw Disturbance Observer

## Overview

This is the latest modified version of Julia port of the Python `Mecanum_MultiComponentFriction_SlipSpin_Roller_ASMC_4` notebook,
built on top of `DifferentialEquations.jl`. It implements the same **high-fidelity numerical simulation**
of a four–mecanum-wheel omnidirectional platform from:

> **"Influence of Dissipative Forces and the Design of Mecanum-Wheels on the Omnidirectional Platform Dynamics"**  
> B.I. Adamov & G.R. Saypulaev, 2021.

### What this notebook does

1. Defines the full 21-D dynamical model (platform, wheel, and roller dynamics), accounting for the finite
   number of rollers per wheel and roller–ground contact geometry.
2. Implements **multicomponent contact friction** (Eq. 5) coupling sliding and spin through the contact-spot
   radius `χ`. When `χ = 0` it reduces to 2D Coulomb friction.
3. Uses a full **Adaptive Sliding Mode Controller (ASMC)** with dynamic sliding-surface gains `λ(e)`,
   equivalent + switching control, smooth adaptive gains, and `tanh` torque saturation — matching the
   Python reference cell-for-cell.
4. Generates training data for the PINN by logging states, controls, and computed friction forces.

### State vector (35-D)

Julia uses 1-based indexing, so:

```
[1]     Vx                  platform longitudinal velocity (body frame)
[2]     Vy                  platform lateral velocity (body frame)
[3]     ψ̇ (psi_dot)        platform yaw rate
[4]     ψ (psi)             platform heading
[5..8]  θ₁..θ₄              wheel rotation angles
[9..12] ω₁..ω₄              wheel angular velocities
[13..16] γ₁..γ₄             roller angular velocities  (NOT used by PINN)
[17..19] K_x, K_y, K_ψ      ASMC adaptive gains
[20..21] x₀, y₀             global position (for tracking error)
[22..25] zx₁..zx₄           LuGre translational bristle (x)
[26..29] zy₁..zy₄           LuGre translational bristle (y)
[30..33] zs₁..zs₄           LuGre spin bristle
[34..36] ζ_x, ζ_y, ζ_psi    Disturbance-observer (DOB) auxiliary states
[37..39] v_hat for observers in x, y and psi for supertwist mode when enabled else 0
```

### Why Julia here

Three concrete wins over the scipy/Python version:

- **Speed.** `Rodas5P`/`QNDF`/`RadauIIA5`,`TRBDF2` in `OrdinaryDiffEq.jl` is a well-engineered stiff solver with
  PI step control, and the RHS compiles to native code — expect 10–30× over `scipy.solve_ivp(method='Radau')`
  on the same tolerances.
- **Autodiff Jacobian.** We pass an `ODEFunction` with `AutoForwardDiff()` so the solver gets an exact
  Jacobian for free instead of finite-differencing it every Newton iteration.
- **Efficient storage.** We write results as **Arrow (Feather v2)** — columnar, compressed, cross-language
  — and also dump a native **JLD2** snapshot for fast Julia-side reloading. Arrow files are ~5–10× smaller
  than CSV and ~20× faster to load into Python's `pyarrow`/`pandas` for PINN training.


### Parameterization (papermill-equivalent)

This cell is tagged `parameters`. External drivers (a Julia `Data_Generation.jl` or `papermill`
using `IJulia` + `papermill-julia`) can inject different values for data-generation sweeps.


In [ ]:
# PARAMETERS — override from outside via papermill or by editing here

# ---- Trajectory selection (profile pipeline) -------------------------------
CONFIG_DIR   = "trajectory_files_run_0p3_main"
PROFILE_SET  = ["octagon.toml", "long_circle.toml", "spin_creep.toml",
                "coupled_vomega.toml", "multisine_75percent_cap.toml", "spiral_orbit.toml", "ellipse.toml"]
PROFILE_FILE = "octagon_mu_0p3.toml"   # nothing → random pick from PROFILE_SET (seeded);
                         # or e.g. "octagon.toml" to pin the profile
COMBO_IDX    = 23  # nothing → random combo row; or an Int to reproduce
                         # the EXACT sweep job (profile, combo) from the driver
PICK_SEED    = 123

# ---- Physics point for this notebook run ------------------------------------
# (The sweep DRIVER iterates the [sweep] lists in configs/base.toml instead.)
mu_friction   = 0.3          # Coulomb friction coefficient
friction_case = 1            # 1 = standard viscous pair, 2 = low viscous pair
write_data    = false        # notebook-only gate: stream this run to Arrow/JLD2
output_dir    = "SimulationDataSlipSpin_Julia_LuGre"

# --------------------------------------------------------------------------
# Contact-friction model selection (steady-state LuGre; see "Friction model" cell)
#   :lugre_uncoupled — DECOUPLED denominators (published anchor)
#   :lugre_adamov    — Adamov COUPLED denominators + Mindlin slip-annulus ramp
# --------------------------------------------------------------------------
friction_model = :lugre_adamov    # used when compare_models == false
compare_models = true             # true → run BOTH models per χ, side by side

# χ values in / out of the PINN training distribution
train_chi_values   = [0.005]
heldout_chi_values = []

# --------------------------------------------------------------------------
# MIMO disturbance observer — unchanged from the previous parameters cell
# --------------------------------------------------------------------------
use_dob     = false
omega_o_x   = 0.0
omega_o_y   = 0.0
omega_o_psi = 6π

dob_kind = :super_twisting     # :super_twisting | :linear
k1_x = 0.0;    k2_x = 0.0
k1_y = 0.0;    k2_y = 0.0
k1_psi = 15.0; k2_psi = 80.0
eps_obs = 1e-2

vel_asmc = true    # mixed relative-degree controller = deployment + training controller

### Library Imports

- `OrdinaryDiffEq`, `DiffEqCallbacks`: the stiff-solver core from SciML. `Rodas5P` is a 5th-order
  Rosenbrock method that handles the mild stiffness here very well; `QNDF` (a BDF) is an
  alternate choice for longer runs. Automatic differentiation of the Jacobian via `ForwardDiff`
  is enabled by default for stiff solvers in OrdinaryDiffEq — no explicit setup is needed.
- `Symbolics`: symbolic derivatives of reference trajectories (replaces SymPy).
- `StaticArrays`: stack-allocated small vectors for the per-wheel arrays — avoids heap allocations
  in the hot RHS loop.
- `Arrow`, `JLD2`, `DataFrames`, `CSV`: data output. Arrow is the primary columnar format;
  JLD2 is the Julia-native snapshot.
- `ProgressMeter`: live progress bar during `solve`.


In [ ]:
using Revise
using LinearAlgebra
using OrdinaryDiffEq
using DiffEqCallbacks
using StaticArrays
using DataFrames
using Arrow
using JLD2
using Plots
using Printf
using ProgressMeter
using Base.Threads: @spawn

# Reproducibility
import Random
Random.seed!(0)

# ---------------------------
# Pipeline modules — keep profiles.jl / datastore.jl / configs/ beside the notebook
# ---------------------------
includet("profiles.jl");  using .Profiles
includet("datastore.jl"); using .DataStore
println("Profiles: ", join(sort(collect(keys(Profiles.BUILDERS))), ", "))


### Platform Physical Parameters (`PlatformParams`)

Matches the KUKA youBot parameters from the reference paper. Everything is packed into an immutable
`struct` with `SVector`/`SMatrix` fields so the hot-path RHS allocates nothing.

- **Geometry**: half-dimensions `h=0.235 m`, `l=0.15 m`; wheel radius `R=0.05 m`; roller-axle
  distance `Rₐ=0.0355 m`.
- **Mass & inertia**: platform `m=30 kg`, wheel `m_w=1.4 kg`, `J_w=5.87e-3`, `J_r=1e-6`.
- **Friction cases**: two `(p₁, p₂)` pairs for viscous friction in the base-wheel and wheel-roller joints.
- **Normal forces** `Nᵢ`: per-wheel static distribution accounting for COM offset `(aX, aY)`.
- **Mass matrix** `M` (3×3) and its precomputed inverse: platform translational+rotational coupling.


In [ ]:
# ---------------------------
# Platform geometry and parameters
# ---------------------------
struct PlatformParams
    # Geometry
    h::Float64          # half-length (m)
    l::Float64          # half-width (m)
    R::Float64          # wheel outer radius (m)
    Ra::Float64         # roller axle distance (m)
    # Masses/inertias
    m::Float64
    m_wheel::Float64
    J_wheel::Float64
    J_roller::Float64
    ms::Float64         # total mass
    Is::Float64         # platform moment of inertia
    # Viscous friction cases
    p1_case1::Float64
    p2_case1::Float64
    p1_case2::Float64
    p2_case2::Float64
    # Friction
    f_coulomb::Float64
    N_total::Float64
    rollers_per_wheel::Int
    # Per-wheel static fields (SVector for stack allocation)
    delta::SVector{4,Float64}       # roller axis angle
    wc_x::SVector{4,Float64}        # wheel center X
    wc_y::SVector{4,Float64}        # wheel center Y
    aX::Float64                     # COM offset X
    aY::Float64                     # COM offset Y
    N_per_roller::SVector{4,Float64}
    M_inv::SMatrix{3,3,Float64,9}        # plant body mass matrix (no wheel reflection)
    M_aug::SMatrix{3,3,Float64,9}        # augmented mass matrix (with wheel reflection)
    M_aug_inv::SMatrix{3,3,Float64,9}    # inverse of the augmented mass matrix
    Max_torque::Float64
end

function PlatformParams(base::AbstractDict; mu_friction::Float64 = mu_friction)
    geo = base["platform"]["geometry"];  mas = base["platform"]["mass"]
    com = base["platform"]["com_offset"]; vis = base["platform"]["viscous"]
    con = base["platform"]["contact"]
    h  = geo["h"];  l = geo["l"];  R = geo["R"];  Ra = geo["Ra"]
    m  = mas["m"];  m_wheel = mas["m_wheel"]
    J_wheel = mas["J_wheel"];  J_roller = mas["J_roller"]
    ms = m + 4.0 * m_wheel                       # derived — stays in code
    Is = mas["Is"]
    p1_case1 = vis["p1_case1"]; p2_case1 = vis["p2_case1"]
    p1_case2 = vis["p1_case2"]; p2_case2 = vis["p2_case2"]
    f_coulomb = mu_friction                      # sweep variable: kwarg wins
    N_total = m * 9.81
    rollers_per_wheel = con["rollers_per_wheel"]
    delta = SVector(-π/4, π/4, π/4, -π/4)        # structural, not config
    wc_x  = SVector(h, h, -h, -h);  wc_y = SVector(l, -l, l, -l)
    aX = com["aX"];  aY = com["aY"]

    N_per_roller = SVector(
        N_total/4 * (1 + aX/h + aY/l) + m_wheel * 9.81,
        N_total/4 * (1 + aX/h - aY/l) + m_wheel * 9.81,
        N_total/4 * (1 - aX/h + aY/l) + m_wheel * 9.81,
        N_total/4 * (1 - aX/h - aY/l) + m_wheel * 9.81
    )
    M_mat = @SMatrix [ ms       0.0     -m*aY;
                       0.0      ms       m*aX;
                      -m*aY     m*aX     Is   ]
    M_inv = inv(M_mat)
    # MIMO DOB uses the augmented mass matrix that the equivalent control already inverts
    # (m_tilde and I_psi_aug pick up the wheel rotational inertia reflected to the body).
    m_tilde   = ms + 4 * J_wheel / R^2
    I_psi_aug = Is + 4 * (l + h)^2 / R^2 * J_wheel
    M_aug = @SMatrix [ m_tilde   0.0       -m*aY ;
                       0.0       m_tilde    m*aX ;
                      -m*aY      m*aX       I_psi_aug ]
    M_aug_inv = inv(M_aug)
    Max_torque = 10.0
    PlatformParams(h, l, R, Ra, m, m_wheel, J_wheel, J_roller, ms, Is,
                   p1_case1, p2_case1, p1_case2, p2_case2, f_coulomb,
                   N_total, rollers_per_wheel, delta, wc_x, wc_y,
                   aX, aY, N_per_roller, M_inv, M_aug, M_aug_inv, Max_torque)
end


### ASMC Controller Configuration

Same control-tuning fields as the Python version — dynamic sliding-surface convergence rate
`λ(e)` via a Gaussian bump, adaptive gain cap `K_max`, and boundary-layer thickness `eps` for
the `tanh` smooth-sign.


In [ ]:
# ---------------------------
# ASMC Controller Configuration  (per-axis gains)
#   λ ordering  ψ > y > x   — closed-loop bandwidth / authority.
#   γ, K_max    the MIMO DOB carries the lumped disturbance on all three axes, so γ_y
#               and γ_psi are reduced when DOB is active to avoid the adaptive gain and
#               the DOB double-counting the same disturbance. Y > X because My_eq carries
#               the extra 8·p2/(R-Rd)² roller-viscous drag term that Mx_eq does not.
# ---------------------------
Base.@kwdef struct ASMCParams
    gamma_x::Float64     = 8.0      # adaptation rate (X)
    gamma_y::Float64     = use_dob ? 12.0 : 15.0      # adaptation rate (Y)
    gamma_psi::Float64   = use_dob ? 12.0 : 25.0     # adaptation rate (heading) — reduced when DOB active
    eps::Float64         = 0.0175    # boundary-layer thickness (X/Y)
    eps_psi::Float64     = 0.08     # boundary-layer thickness (yaw residual) — slightly thicker
    K_max_x::Float64     = 60.0     # gain saturation (X)
    K_max_y::Float64     = 80.0     # gain saturation (Y)
    K_max_psi::Float64   = 100.0    # gain saturation (heading)
    lam_x_min::Float64   = 0.1
    lam_x_max::Float64   = 1.5
    lam_y_min::Float64   = 0.2
    lam_y_max::Float64   = 2.5
    lam_psi_min::Float64 = 0.5
    lam_psi_max::Float64 = 5.0
    mu_xy::Float64       = 25.0     # sharpness of λ(e) curve for X/Y
    mu_psi::Float64      = 100.0    # sharpness of λ(e) curve for heading
    sigma_psi::Float64   = 0.25
    sigma_x::Float64   = 0.5
    sigma_y::Float64   = 0.25
    decay_k::Float64   = 0.25
    K_x0::Float64   = 5.0
    K_y0::Float64   = 5.0
    K_psi0::Float64   = 20.0
end

# ---------------------------
# MIMO disturbance-observer (reduced-order DOB) configuration.
#   Nominal:   v̇ = M_aug⁻¹·F_p + δ       (v=[Vx,Vy,ψ̇]ᵀ; δ=[δ_x,δ_y,δ_ψ]ᵀ)
#   Estimate:  δ̂ = ζ + ω_o·v             (ζ ∈ R³  at state[34], state[35], state[36])
#   ODE:       ζ̇ = -ω_o·ζ - ω_o²·v - ω_o·M_aug⁻¹·F_p_applied
#   Compens:   ΔM_c = -diag(R,R,1)·M_aug·δ̂   added to the equivalent-control wrench
#   F_p_applied recovered from saturated wheel torques via the un-map (see asmc_torques).
#   One scalar bandwidth ω_o serves all three channels (Ω = ω_o·I₃).
# ---------------------------
Base.@kwdef struct ESOParams
       kind::Symbol         = :super_twisting   # :super_twisting | :linear
       # --- reduced-order linear DOB bandwidths (used when kind == :linear) ---
       omega_o_x::Float64   = 20.0
       omega_o_y::Float64   = 20.0
       omega_o_psi::Float64 = 20.0
       # --- super-twisting observer gains (used when kind == :super_twisting) ---
       #   v̂̇ = a_nom + δ̂ + k₁·φ₁(v−v̂);  δ̂̇ = k₂·φ₂(v−v̂)
       k1_x::Float64        = 0.0
       k2_x::Float64        = 0.0
       k1_y::Float64        = 0.0
       k2_y::Float64        = 0.0
       k1_psi::Float64      = 15.0
       k2_psi::Float64      = 80.0
       eps_obs::Float64     = 1e-2   # C∞ boundary layer for φ₁, φ₂
       enable::Bool         = true
   end

### Roller Contact Geometry

The `sawtooth_approx` function models the periodic roller handoff as each wheel rotates. With
**12 rollers per wheel** (up from the paper's 6, for better numerical fidelity of the tanh-smoothed
sawtooth), the contacting-roller angle `φ̃ᵢ` varies periodically within ±15° (±π/12 rad).

We use the **15-term Lanczos-smoothed Fourier series** from the Python v4 notebook — it's identical
coefficients, just rewritten in Julia. The smooth analytic form is essential: an ideal sawtooth would
make the ODE RHS discontinuous and kill adaptive step-size control.

The lateral contact displacement `ΔYᵢ = Rₐ tan(δᵢ) tan(φ̃ᵢ)` is the geometric origin of the
mecanum wheel's omnidirectional behavior.


In [ ]:
# ---------------------------
# Smoothed sawtooth: two implementations
#
# SAWTOOTH = :tanh     → tanh-smoothed peak: fast (~5× over Fourier), localized error at handoff.
# SAWTOOTH = :fourier  → 14-term Lanczos-smoothed Fourier sum (matches Python v4 notebook).
#
# Period = 30° = π/6 rad, output in radians, range ±π/12 (±15°).
# ---------------------------
const SAWTOOTH = :tanh            # switch to :fourier to reproduce Python v4 exactly
const TANH_K   = 60.0             # tanh steepness at the peaks; 20–60 is a reasonable range

# --- tanh-peak version (preferred for speed) ---
# Idea: take the *ideal* sawtooth, then soften only the peaks.
# s_ideal(θ) = ((θ + π/12) mod π/6) - π/12,  which equals  atan(tan(6θ))/6  (single branch).
# We write it as (1/6)·atan(tan(6θ)) and replace the unbounded `tan` with a bounded
# tanh-shaped peak so the function stays C∞:  sin(12θ) / (β + cos(12θ)) form.
@inline function sawtooth_tanh(θ)
    # Bounded, C∞ approximation with a single `atan` + `sin` + `cos`.
    # On the linear ramp this agrees with the ideal sawtooth to ~1e-4 rad for TANH_K ≥ 40.
    s = sin(12θ)
    c = cos(12θ)
    return atan(TANH_K * s, TANH_K * c + 1) / 12  # 2-arg atan -> smooth branch switch
end

# --- 14-term Lanczos-smoothed Fourier sum (reference / backward compatibility) ---
@inline function sawtooth_fourier(θ)
     (0.165450866601201 * sin( 12*θ)
    - 0.080917684102647 * sin( 24*θ)
    + 0.051971626877147 * sin( 36*θ)
    - 0.036960991377466 * sin( 48*θ)
    + 0.027566444771090 * sin( 60*θ)
    - 0.021022964684463 * sin( 72*θ)
    + 0.016151334251121 * sin( 84*θ)
    - 0.012365865286014 * sin( 96*θ)
    + 0.009343539859761 * sin(108*θ)
    - 0.006891611192772 * sin(120*θ)
    + 0.004887403818508 * sin(132*θ)
    - 0.003248226679822 * sin(144*θ)
    + 0.001915211458051 * sin(156*θ)
    - 0.000844137074496 * sin(168*θ))
end

# Dispatch once at include time — no per-call branch cost.
const sawtooth_approx = SAWTOOTH === :tanh ? sawtooth_tanh : sawtooth_fourier
if !write_data
    # Quick comparison plot
    let θ = range(0, 2π; length=2000)
        y_tanh    = sawtooth_tanh.(θ)
        #y_fourier = sawtooth_fourier.(θ)
        y_ideal   = @. mod(rad2deg(θ) + 15, 30) - 15
        plt = plot(rad2deg.(θ), rad2deg.(y_tanh);    label="tanh-peak", lw=2)
        #plot!(plt, rad2deg.(θ), rad2deg.(y_fourier); label="Lanczos-Fourier (N=14)", lw=2, ls=:dash)
        plot!(plt, rad2deg.(θ), y_ideal;             label="Ideal sawtooth", ls=:dot,
            xlabel="θ (deg)", ylabel="φ̃ (deg)", title="Smoothed sawtooth — peak vs Fourier")
        display(plt)
        #Plot error as well
        plt2 = plot(rad2deg.(θ), deg2rad.(y_ideal)-y_tanh; label="tanh", lw=2, xlabel="θ (deg)", ylabel="Error (rad)", xlims=(0,60))
        display(plt2)
    end
end

### Multicomponent Contact Friction Model — two dynamic-LuGre variants

This notebook carries **two dynamic LuGre friction laws**, selected by `friction_model`
(or both at once via `compare_models = true`). Each wheel now carries a **bristle state**
that the solver integrates: $z_x, z_y$ (translational) and $z_s$ (spin) — **3 per wheel, 12 total**,
so the state vector is **33-D** (`u[22:33]`; layout below).

**Stribeck coefficient** (shared):
$$g(s) = \mu_c\,[\,1 + (r_s-1)e^{-(s/v_{str})^2}\,],\qquad r_s=\mu_s/\mu_c.$$

**Dynamic LuGre** per wheel (load-normalised; $\sigma_0$ [1/m], $\sigma_1$ [s/m]):
$$\dot{\mathbf z}=\mathbf V_p-\frac{\sigma_0\,s_t}{g(s_t)}\mathbf z,\quad
  \mathbf F=-N(\sigma_0\mathbf z+\sigma_1\dot{\mathbf z}+\sigma_2\mathbf V_p);\qquad
  \dot z_s=\omega_z-\frac{\sigma_{0s}s_s}{g(s_s)}z_s,\quad
  M_z=-N\chi^2(\sigma_{0s}z_s+\sigma_{1s}\dot z_s+\sigma_{2s}\omega_z).$$

The **only** structural difference between the two models is the effective mean local slip $s_t,s_s$:

| | translational $s_t$ | spin $s_s$ |
|---|---|---|
| **`:lugre_uncoupled`** (anchor) | $\lVert V_p\rVert$ | $\tfrac{16}{3\pi}\lvert\omega_z\rvert\chi$ |
| **`:lugre_adamov`** | $f_{slip}\,\tfrac{8}{3\pi}\lvert\omega_z\rvert\chi+\lVert V_p\rVert$ | $\tfrac{16}{3\pi}\lvert\omega_z\rvert\chi+5\lVert V_p\rVert$ |

At steady state $\mathbf z_{ss}=g\,\mathbf V_p/(\sigma_0 s_t)$ ⟹ $\mathbf F_{ss}=-N g(s_t)\mathbf V_p/s_t$, so at gross
sliding ($g\to\mu_c$, $f_{slip}\to1$) **`:lugre_adamov` collapses onto the published Adamov law** exactly.

**Mindlin ramp — now genuinely state-based.** Because $z$ is a state, the slip fraction is read directly:
$$f_{slip}=1-\Bigl(1-\tfrac{\lVert z_{xy}\rVert}{\delta^*}\Bigr)^{2/3},\qquad \delta^*=\mu_s/\sigma_0,$$
no fixed-point needed. It rides on the spin→translation coupling only; full stick ⟹ $f_{slip}\to0$,
gross sliding ⟹ $f_{slip}\to1$. Set `use_mindlin=false` for plain Adamov coupling.

**Regime behaviour.** **Model 1** has *no Adamov coupling in any regime* (incl. gross sliding).
**Model 2** has *full Adamov coupling, fully active in gross sliding* ($f_{slip}\to1$); Mindlin only
attenuates the spin→translation coupling in microslip; the $5\lVert V_p\rVert$ term is on throughout.
⇒ the two models **do not converge at gross sliding** when $\omega_z\neq0,\chi>0$.

**State layout addition:** `u[22:25]`=$z_{x,1..4}$, `u[26:29]`=$z_{y,1..4}$, `u[30:33]`=$z_{s,1..4}$.


In [ ]:
# ---------------------------
# Contact friction. The ODE uses the DYNAMIC LuGre (bristle states integrated by the solver).
# `multicomponent_friction` (Adamov static) and `lugre_ss_friction` (steady-state) are kept as
# references only; they are NOT used in the RHS.
# ---------------------------

# (reference) original Adamov static law
# @inline function multicomponent_friction(f::Real, N::Real, chi::Real,
#                                          w_z::Real, Vpx::Real, Vpy::Real; eps::Real = 1e-6)
#     Vp = sqrt(Vpx^2 + Vpy^2 + eps^2); awz = sqrt(w_z^2 + eps^2)
#     fs = f * tanh(200 * Vp)
#     Fx = -fs * N * Vpx / ((8/(3π))*awz*chi + Vp)
#     Fy = -fs * N * Vpy / ((8/(3π))*awz*chi + Vp)
#     Mz = -fs * N * (chi^2 * w_z) / ((16/(3π))*awz*chi + 5*Vp)
#     return Fx, Fy, Mz
# end

# --- LuGre / Stribeck parameters (CALIBRATE to your polyurethane rollers) ---
Base.@kwdef struct LuGreParams
    sigma0::Float64   = 1.64e3    # translational bristle stiffness [1/m]
    sigma1::Float64   = 1.6    # translational micro-damping     [s/m]
    sigma2::Float64   = 0.0      # translational viscous           [s/m]  (0: handled by p1/p2)
    sigma0_s::Float64 = 1.09e3    # spin bristle stiffness          [1/m]
    sigma1_s::Float64 = 1.1   # spin micro-damping              [s/m]
    sigma2_s::Float64 = 0.0      # spin viscous
    stiction_ratio::Float64 = 1.1  # μs/μc  (1.0 disables Stribeck dip)
    v_str::Float64    = 0.01     # translational Stribeck velocity [m/s]
    w_str::Float64    = 0.01     # spin Stribeck scale (mean local slip) [m/s]
    use_mindlin::Bool = true     # state-based Mindlin ramp on spin→translation coupling
    mindlin_iters::Int = 2       # (only used by the steady-state reference fn below)
    eps_reg::Float64  = 1e-4     # regularisation [m/s]
end

@inline stribeck_g(s, mu_c, ratio, vs) = mu_c * (1 + (ratio - 1) * exp(-(s / vs)^2))
coupling_of(fm::Symbol) = fm === :lugre_adamov ? :adamov : :uncoupled

# DYNAMIC LuGre: returns forces/torque AND the bristle derivatives (żx, ży, żs).
@inline function lugre_dyn_rates(lg::LuGreParams, coupling::Symbol,
                                 f::Real, N::Real, chi::Real,
                                 w_z::Real, Vpx::Real, Vpy::Real,
                                 zx::Real, zy::Real, zs::Real)
    er  = lg.eps_reg
    Vp  = sqrt(Vpx^2 + Vpy^2 + er^2)
    awz = sqrt(w_z^2 + er^2)
    c_t = (8/(3π)) * awz * chi             # gross-sliding spin → translation coupling

    if coupling === :adamov
        if lg.use_mindlin
            znorm = sqrt(zx^2 + zy^2 + 1e-18)
            dstar = (lg.stiction_ratio * f) / lg.sigma0     # δ* = μs/σ0 (breakaway deflection)
            sfrac = clamp(znorm / dstar, 0.0, 1.0)
            b     = max(1 - sfrac, 1e-9)
            fsl   = 1 - b^(2/3)
        else
            fsl = 1.0
        end
        s_t = fsl * c_t + Vp
        s_s = (16/(3π)) * awz * chi + 5 * Vp
    else  # :uncoupled — no Adamov coupling in any regime
        s_t = Vp
        s_s = (16/(3π)) * awz * chi + er
    end

    g_t = stribeck_g(s_t, f, lg.stiction_ratio, lg.v_str)
    g_s = stribeck_g(s_s, f, lg.stiction_ratio, lg.w_str)

    # bristle derivatives (integrated by the solver)
    dzx = Vpx - lg.sigma0   * s_t / g_t * zx
    dzy = Vpy - lg.sigma0   * s_t / g_t * zy
    dzs = w_z - lg.sigma0_s * s_s / g_s * zs

    # forces / spin torque (include the σ1·ż micro-damping term)
    Fx = -N * (lg.sigma0   * zx + lg.sigma1   * dzx + lg.sigma2   * Vpx)
    Fy = -N * (lg.sigma0   * zy + lg.sigma1   * dzy + lg.sigma2   * Vpy)
    Mz = -N * chi^2 * (lg.sigma0_s * zs + lg.sigma1_s * dzs + lg.sigma2_s * w_z)
    return Fx, Fy, Mz, dzx, dzy, dzs
end

# (reference) steady-state algebraic LuGre — bristle eliminated, Mindlin via Picard.
@inline function mindlin_fslip(V, c, iters)
    x = V / (c + V); fs = 1.0
    @inbounds for _ in 1:iters
        b = max(1 - x, 1e-9); fs = 1 - b^(2/3); x = V / (fs * c + V)
    end
    b = max(1 - x, 1e-9); return 1 - b^(2/3)
end
@inline function lugre_ss_friction(lg::LuGreParams, coupling::Symbol,
                                   f::Real, N::Real, chi::Real, w_z::Real, Vpx::Real, Vpy::Real)
    er = lg.eps_reg; Vp = sqrt(Vpx^2+Vpy^2+er^2); awz = sqrt(w_z^2+er^2)
    c_t = (8/(3π))*awz*chi
    if coupling === :adamov
        fsl = lg.use_mindlin ? mindlin_fslip(Vp, c_t, max(lg.mindlin_iters,1)) : 1.0
        s_t = fsl*c_t + Vp; s_s = (16/(3π))*awz*chi + 5*Vp
    else
        s_t = Vp; s_s = (16/(3π))*awz*chi + er
    end
    g_t = stribeck_g(s_t,f,lg.stiction_ratio,lg.v_str)
    g_s = stribeck_g(s_s,f,lg.stiction_ratio,lg.w_str)
    return (-N*g_t*Vpx/s_t, -N*g_t*Vpy/s_t, -N*chi^2*g_s*w_z/s_s)
end

# Default LuGre parameter set.
lugre = LuGreParams()


### Reference Trajectories & Velocity Generator using Profiles

Modified trajectory types (same as Python v4):

1. **Octagon** — ramp - constant-velocity - ramp down motion along an octagon either linear or sine wave
2. **Long Circle** — CW and CCW circles of different radii - different Vxb and lateral acceleration (body frame)
3. **Spin Creep** — Small translational velocity with high yaw rate; pulsed yaw rates to combine yaw acceleration
4. **Spiral** — Constant translational Vxb, Constant omega, and both ramping up (accelerating circle) 
5. **Multisine** — Rich excitation with multisine components along all 3 body fram velocity axes

Conservative velocity/acceleration caps to not overload the friction circle due to commanded velocities. Currently for mu = 0.5 only. Later we will add more.

Different profiles and their available parameters available in custom profiles.jl and a separate toml file for each case with profiles indexed in a long index format for consistency. Each case needs different parameter inputs so this surgical edit made it possible.

In the notebook, it picks a profile (if not specified otherwise fixed) and one of its trajectories at random from relevant toml. In data generation, it sweeps the toml for all trajectories 


In [ ]:
# ---------------------------
# Reference trajectory — built from the profile library.
#   PROFILE_FILE === nothing → seeded random pick across PROFILE_SET
#   PROFILE_FILE set         → that profile; COMBO_IDX pins the exact sweep job
# Publishes via publish! — VelRef -> CURRENT_REF (asmc_torques_vel),
# PosRef -> CURRENT_POSREF (asmc_torques, e.g. the ellipse profile).
# ---------------------------
PICK_RNG = Random.Xoshiro(PICK_SEED)

BASE = load_base(CONFIG_DIR)

REF, TRAJ_CFG, PROFILE_NAME = if PROFILE_FILE === nothing
    pick_and_build(CONFIG_DIR, PROFILE_SET; rng = PICK_RNG)
else
    jobs = enumerate_jobs(CONFIG_DIR, [PROFILE_FILE]; sweep_seed = PICK_SEED)
    idx  = COMBO_IDX === nothing ? Random.rand(PICK_RNG, 1:length(jobs)) : Int(COMBO_IDX)
    job  = jobs[idx]
    ref, cfg = build_job(job)
    publish!(ref)                    # dispatches VelRef/PosRef to the right slot
    (ref, cfg, job.profile)
end

T_total = REF.T_total

PEAK_V = let tt = range(0, T_total; length = 2001)
    hasproperty(REF, :Vx) ? maximum(hypot(REF.Vx(t),  REF.Vy(t))  for t in tt) :
                            maximum(hypot(REF.Vxo(t), REF.Vyo(t)) for t in tt)
end

println("ψ_des(0)=$(round(REF.psi(0.0); digits=4)) rad   peak |V|≈$(round(PEAK_V; digits=3)) m/s")
println("Profile: $PROFILE_NAME  combo=$(get(TRAJ_CFG, "combo_idx", 0))  ",
        "T_total=$(round(T_total; digits=2)) s  tstops=$(length(REF.tstops))")


### ASMC and DOB Helpers

- `smooth_bound(K, K_max)` — a soft ceiling on adaptive gains; smoothly goes from 1 → 0 as
  `K` approaches `K_max`, preventing windup.
- `get_dynamic_lambda(e, ė, λ_min, λ_max, μ)` — a Gaussian `λ(e) = λ_min + (λ_max − λ_min) e^{−μ e²}`
  that increases the sliding-surface convergence rate when close to the target (small `e`) and
  relaxes it when far. Also returns the analytic time derivative `λ̇` needed inside the equivalent
  control.
- `smooth_sat` — a smooth saturator capping values higher than tanh clamp for max. keep n = 3 or 4
- `reg_sqrt and reg_sign` — a ODE solver friendly smooth approximation of square root and absolute value functions in supertwist dob


In [ ]:
@inline function smooth_bound(K, K_max)
    # same 1-0 smooth gate as Python v4
    return 0.5 - 0.5 * tanh(1.0 * (K - (K_max - 2.0)))
end

@inline function get_dynamic_lambda(e, edot, lam_min, lam_max, mu)
    exp_term = exp(-mu * e^2)
    lam = lam_min + (lam_max - lam_min) * exp_term
    lam_dot = -2 * mu * e * edot * (lam_max - lam_min) * exp_term
    return lam, lam_dot
end

@inline smooth_sat(M, L, n::Int = 3) = M / (1 + (M / L)^(2n))^(1 / (2n))
# Smoother but sharper saturation compared to tanh. 

# ---- C∞ regularizations for the super-twisting observer ---------------------
# Smooth everywhere (denominator never hits 0), so RadauIIA5's Newton/error
# estimator and ForwardDiff stay well-behaved as the observer error crosses 0.
@inline reg_sign(x, eps) = x / sqrt(x^2 + eps^2)        # ≈ sign(x)
@inline reg_sqrt(x, eps) = x / (x^2 + eps^2)^0.25       # ≈ |x|^½ · sign(x)

# global_to_local_frame now comes from Profiles (exported) — defining it here
# over the `using .Profiles` binding would throw a method-definition error.


### ASMC Control Law (Full, Smoothed) For Position Tracking

This is the full ASMC — not the simplified feedforward + switching
version that was in the earlier Julia iteration. Steps per call:

1. **Position errors in the body frame** — rotate global `(x₀−x_d, y₀−y_d)` through `ψ`.
2. **Velocity errors** — with cross-coupling to yaw rate through `ψ̇·e_y`, `−ψ̇·e_x` terms.
3. **Dynamic λ** — Gaussian convergence rate in each axis.
4. **Sliding surfaces** — `s = ė + λ·e` (second degree) or `s = e` (first degree) .
5. **Equivalent control** — cancels the known platform dynamics, using the dynamic `λ̇` term.
6. **Switching control** — `−K · tanh(s/ε)` with adaptive `K`.
7. **Map task-space wrench → 4 wheel torques** via the standard mecanum kinematic mapping
   (uses `R/(l+h)` for the yaw lever arm).
8. **Adaptive gain dynamics** — `K̇ = γ · s · tanh(s/ε) · smooth_bound(K, K_max) − c·(K/K_max)³ - σ·(K-Kₘᵢₙ)·exp{k·(1-(s/3ϵ)²)`
   with per-axis cubic pushback to prevent windup near max. Added an exponential decay based kickback to draw K to K_min when switching term needs to
   be small to avoid chatter

### ASMC Control Law (Full, Smoothed) For Velodity Tracking

Mixed order (degree 1 in x/y velocity and 2 in yaw pose) to ensure trajectories dont drift too far away when commanded with only body frame velocities.
Useful when representation in body frame is simpler


In [ ]:
# ---------------------------
# ASMC Control Law (full, smoothed) + MIMO disturbance-observer compensation.
# Returns: Mi_sw, Mi_eq (each SVector{4}), dK (SVector{3}), δ̂ (SVector{3} = [δ̂_x, δ̂_y, δ̂_ψ]).
# The equivalent-control wrench is corrected by ΔM_c = -diag(R,R,1)·M_aug·δ̂.
#   δ̂ = ζ + ω_o·v        with v = [Vx, Vy, ψ̇]ᵀ  and  ζ = [state[34], state[35], state[36]]ᵀ.
# Per-axis compensation (derivation §7, eqs 18-20):
#   ΔM_x   = -R·m̃·δ̂_x + R·m·aY·δ̂_ψ
#   ΔM_y   = -R·m̃·δ̂_y - R·m·aX·δ̂_ψ
#   ΔM_ψ   =    m·aY·δ̂_x - m·aX·δ̂_y - I_ψ·δ̂_ψ 
# ---------------------------
function asmc_torques(state::AbstractVector, t::Real,
                     params::PlatformParams, asmc::ASMCParams, eso::ESOParams)
    # Unpack params
    m, ms, Is, J1 = params.m, params.ms, params.Is, params.J_wheel
    aX, aY = params.aX, params.aY
    p1, p2 = params.p1_case1, params.p2_case1
    l, h, R, Rd = params.l, params.h, params.R, params.Ra
    mu_xy, mu_psi = asmc.mu_xy, asmc.mu_psi

    # Unpack state (Julia 1-based; matches Python v4 layout shifted by +1)
    Vx, Vy, psi_dot, psi = state[1], state[2], state[3], state[4]
    K_x, K_y, K_psi      = state[17], state[18], state[19]
    xo, yo               = state[20], state[21]

    # Disturbance estimate δ̂.  Super-twisting: δ̂ is the integrated observer
    # state directly (state[37:39]).  Linear: δ̂ = ζ + ω_o·v  (state[34:36]).
    v_body    = SVector(Vx, Vy, psi_dot)

    en = 0.0
    delta_hat = 0.0
    if eso.enable
        en = 1.0
        if eso.kind == :super_twisting
            delta_hat = en * SVector(state[37], state[38], state[39])
        else
            zeta_vec  = SVector(state[34], state[35], state[36])
            Omega     = SVector(eso.omega_o_x, eso.omega_o_y, eso.omega_o_psi)
            delta_hat = en * (zeta_vec + Omega .* v_body)
        end
    end
    
    # Effective yaw inertia (also used in the equivalent control; same value lives in M_aug[3,3])
    I_psi = Is + 4*(l + h)^2 / R^2 * J1
    
    ref = current_posref()
    # Positional errors (global frame)
    e_xo = xo - ref.xo(t)
    e_yo = yo - ref.yo(t)

    # Rotate global position errors into the body frame; e_psi uses sin(Δψ) for smooth wrap
    c_psi, s_psi = cos(psi), sin(psi)
    e_x = e_xo * c_psi + e_yo * s_psi
    e_y = -e_xo * s_psi + e_yo * c_psi
    d_psi = psi - ref.psi(t)
    e_psi = 2 * tan(d_psi/2) * (1 + 2 * (1 - cos(d_psi)))

    # Desired velocities in local frame
    Vx_d, Vy_d, omega_d = global_to_local_frame(t, psi, ref.Vxo, ref.Vyo, ref.om)

    # Velocity errors (local frame; coupled through yaw rate)
    edot_x   = Vx      - Vx_d + psi_dot * e_y
    edot_y   = Vy      - Vy_d - psi_dot * e_x
    edot_psi = psi_dot - omega_d
    edash_psi = (sec(d_psi/2))^2 * (3 - 2 * cos(d_psi)) + 4 * tan(d_psi/2) * sin(d_psi)

    # Dynamic λ — per-axis ranges (ψ > y > x)
    lam_x,   lam_dot_x   = get_dynamic_lambda(e_x,   edot_x,   asmc.lam_x_min,   asmc.lam_x_max,   mu_xy)
    lam_y,   lam_dot_y   = get_dynamic_lambda(e_y,   edot_y,   asmc.lam_y_min,   asmc.lam_y_max,   mu_xy)
    lam_psi, lam_dot_psi = get_dynamic_lambda(e_psi, edash_psi * edot_psi, asmc.lam_psi_min, asmc.lam_psi_max, mu_psi)

    # Sliding surfaces
    s_x   = edot_x   + lam_x   * e_x
    s_y   = edot_y   + lam_y   * e_y
    s_psi = edot_psi + lam_psi * e_psi

    # Smooth signum (per-axis boundary layer; yaw slightly thicker)
    ss_x   = tanh(s_x   / asmc.eps)
    ss_y   = tanh(s_y   / asmc.eps)
    ss_psi = tanh(s_psi / asmc.eps_psi)

    # Task-space switching wrench
    Mx_sw   = -K_x   * ss_x
    My_sw   = -K_y   * ss_y
    Mpsi_sw = -K_psi * ss_psi

    # Equivalent control — desired body-frame acceleration minus λ terms
    Ax_des, Ay_des, alpha_des = global_to_local_frame(t, psi, ref.Axo, ref.Ayo, ref.al)
    alpha_eq = alpha_des - lam_dot_psi * e_psi - lam_psi * edot_psi * edash_psi
    Ax_eq    = Ax_des    - lam_dot_x   * e_x   - lam_x   * edot_x   + psi_dot * (Vy_d - edot_y) - alpha_eq * e_y 
    Ay_eq    = Ay_des    - lam_dot_y   * e_y   - lam_y   * edot_y   - psi_dot * (Vx_d - edot_x) + alpha_eq * e_x

    # Base equivalent-control wrench (inverse-dynamics; no DOB compensation yet)
    Mx_eq = R * ((ms + 4*J1/R^2) * Ax_eq
                 - ms * psi_dot * Vy
                 - m  * aY * alpha_eq
                 - m  * aX * psi_dot^2
                 + 4  * p1 * Vx / R^2)

    My_eq = R * ((ms + 4*J1/R^2) * Ay_eq
                 + ms * psi_dot * Vx
                 + m  * aX * alpha_eq
                 - m  * aY * psi_dot^2
                 + (4 * p1 / R^2 + 8 * p2 / (R - Rd)^2) * Vy)

    M_psi_eq = (I_psi * alpha_eq
                - m * aY * (Ax_eq - psi_dot * Vy)
                + m * aX * (Ay_eq + psi_dot * Vx)
                + (4*p1*(l+h)^2/R^2 + 8*p2*h^2/(R-Rd)^2) * psi_dot)

    # MIMO DOB compensation  ΔM_c = -diag(R, R, 1)·M_aug·δ̂   (derivation eq 17, equivalently 18-20)
    M_aug_dh = params.M_aug * delta_hat
    Mx_eq    -= R * M_aug_dh[1]
    My_eq    -= R * M_aug_dh[2]
    M_psi_eq -=     M_aug_dh[3]

    # Map task-space wrenches to 4 wheel torques (mecanum O-configuration)
    lever = R / (l + h)
    M1_sw = 0.25 * (Mx_sw - My_sw - lever * Mpsi_sw)
    M2_sw = 0.25 * (Mx_sw + My_sw + lever * Mpsi_sw)
    M3_sw = 0.25 * (Mx_sw + My_sw - lever * Mpsi_sw)
    M4_sw = 0.25 * (Mx_sw - My_sw + lever * Mpsi_sw)

    M1_eq = 0.25 * (Mx_eq - My_eq - lever * M_psi_eq)
    M2_eq = 0.25 * (Mx_eq + My_eq + lever * M_psi_eq)
    M3_eq = 0.25 * (Mx_eq + My_eq - lever * M_psi_eq)
    M4_eq = 0.25 * (Mx_eq - My_eq + lever * M_psi_eq)

    Mi_sw = SVector(M1_sw, M2_sw, M3_sw, M4_sw)
    Mi_eq = SVector(M1_eq, M2_eq, M3_eq, M4_eq)

    # Smooth adaptive-gain dynamics, with per-axis cubic pushback & per-axis K_max
    base_dK_x   = asmc.gamma_x   * (s_x   * ss_x)
    base_dK_y   = asmc.gamma_y   * (s_y   * ss_y)
    base_dK_psi = asmc.gamma_psi * (s_psi * ss_psi)
    dK_x   = base_dK_x   * smooth_bound(K_x,   asmc.K_max_x)   - 0.1 * (K_x   / asmc.K_max_x)^3 - asmc.sigma_x   * (K_x   - asmc.K_x0   * 0.95) * exp(asmc.decay_k*(1 - s_x^2  /(9 * asmc.eps_psi^2)))
    dK_y   = base_dK_y   * smooth_bound(K_y,   asmc.K_max_y)   - 0.3 * (K_y   / asmc.K_max_y)^3 - asmc.sigma_y   * (K_y   - asmc.K_y0   * 0.95) * exp(asmc.decay_k*(1 - s_y^2  /(9 * asmc.eps_psi^2)))
    dK_psi = base_dK_psi * smooth_bound(K_psi, asmc.K_max_psi) - 0.5 * (K_psi / asmc.K_max_psi)^3 - asmc.sigma_psi * (K_psi - asmc.K_psi0 * 0.95) * exp(asmc.decay_k*(1 - s_psi^2/(9 * asmc.eps_psi^2)))

    return Mi_sw, Mi_eq, SVector(dK_x, dK_y, dK_psi), delta_hat
end


In [ ]:
# ---------------------------
# Velocity-tracking ASMC (relative degree 1). Same signature & return tuple as
# asmc_torques, so it drops into the injected controller call in the RHS.
# Surface = body-frame velocity error (no position error, no dynamic λ); the
# equivalent control uses the body-frame accel feedforward directly.
# Requires body-frame reference getters from the spiral profiles:
#   ref.Vx, ref.Vy, ref.Wz                    (velocity, ref = current_ref())
#   ref.Ax, ref.Ay, ref.al                    (accel feedforward)
# Implemented mixed degree control as tracking pose without unbounded error is important for 
# ---------------------------
function asmc_torques_vel(state::AbstractVector, t::Real,
                          params::PlatformParams, asmc::ASMCParams, eso::ESOParams)
    # Unpack params
    m, ms, Is, J1 = params.m, params.ms, params.Is, params.J_wheel
    aX, aY = params.aX, params.aY
    p1, p2 = params.p1_case1, params.p2_case1
    l, h, R, Rd = params.l, params.h, params.R, params.Ra

    # Unpack state
    Vx, Vy, psi_dot, psi = state[1], state[2], state[3], state[4]
    K_x, K_y, K_psi = state[17], state[18], state[19]

    # Disturbance estimate δ̂ (identical to position controller)
    v_body = SVector(Vx, Vy, psi_dot)
    
    en = 0.0
    delta_hat = 0.0
    if eso.enable
        en = 1.0
        if eso.kind == :super_twisting
            delta_hat = en * SVector(state[37], state[38], state[39])
        else
            zeta_vec  = SVector(state[34], state[35], state[36])
            Omega     = SVector(eso.omega_o_x, eso.omega_o_y, eso.omega_o_psi)
            delta_hat = en * (zeta_vec + Omega .* v_body)
        end
    end

    I_psi = Is + 4*(l + h)^2 / R^2 * J1

    ref = current_ref()
    
    # ---- velocity-tracking surface (relative degree 1; body frame in translation, degree 2 in yaw) ----
    Vx_d    = ref.Vx(t)
    Vy_d    = ref.Vy(t)
    omega_d = ref.Wz(t)

    d_psi = psi - ref.psi(t)
    e_psi = 2 * tan(d_psi/2) * (1 + 2 * (1 - cos(d_psi)))
    edash_psi = (sec(d_psi/2))^2 * (3 - 2 * cos(d_psi)) + 4 * tan(d_psi/2) * sin(d_psi)
    edot_psi  = edash_psi * (psi_dot - ref.Wz(t))
    lam_psi, lam_dot_psi = get_dynamic_lambda(e_psi, edot_psi, asmc.lam_psi_min, asmc.lam_psi_max, asmc.mu_psi)
    
    s_x   = Vx      - Vx_d
    s_y   = Vy      - Vy_d
    s_psi = edot_psi + lam_psi * e_psi

    # Smooth signum (per-axis boundary layer; yaw slightly thicker)
    ss_x   = tanh(s_x   / asmc.eps)
    ss_y   = tanh(s_y   / asmc.eps)
    ss_psi = tanh(s_psi / asmc.eps_psi)

    # Task-space switching wrench
    Mx_sw   = -K_x   * ss_x
    My_sw   = -K_y   * ss_y
    Mpsi_sw = -K_psi * ss_psi

    # Equivalent control — body-frame desired-acceleration feedforward (no λ terms in translation)
    Ax_eq    = ref.Ax(t)
    Ay_eq    = ref.Ay(t)
    alpha_eq = ref.al(t) - lam_dot_psi * e_psi - lam_psi * edot_psi * edash_psi

    # Base equivalent-control wrench (inverse-dynamics; identical to position controller)
    Mx_eq = R * ((ms + 4*J1/R^2) * Ax_eq
                 - ms * psi_dot * Vy
                 - m  * aY * alpha_eq
                 - m  * aX * psi_dot^2
                 + 4  * p1 * Vx / R^2)

    My_eq = R * ((ms + 4*J1/R^2) * Ay_eq
                 + ms * psi_dot * Vx
                 + m  * aX * alpha_eq
                 - m  * aY * psi_dot^2
                 + (4 * p1 / R^2 + 8 * p2 / (R - Rd)^2) * Vy)

    M_psi_eq = (I_psi * alpha_eq
                - m * aY * (Ax_eq - psi_dot * Vy)
                + m * aX * (Ay_eq + psi_dot * Vx)
                + (4*p1*(l+h)^2/R^2 + 8*p2*h^2/(R-Rd)^2) * psi_dot)
                

    # MIMO DOB compensation  ΔM_c = -diag(R, R, 1)·M_aug·δ̂
    M_aug_dh = params.M_aug * delta_hat
    Mx_eq    -= R * M_aug_dh[1]
    My_eq    -= R * M_aug_dh[2]
    M_psi_eq -=     M_aug_dh[3]

    # Map task-space wrenches to 4 wheel torques (mecanum O-configuration)
    lever = R / (l + h)
    M1_sw = 0.25 * (Mx_sw - My_sw - lever * Mpsi_sw)
    M2_sw = 0.25 * (Mx_sw + My_sw + lever * Mpsi_sw)
    M3_sw = 0.25 * (Mx_sw + My_sw - lever * Mpsi_sw)
    M4_sw = 0.25 * (Mx_sw - My_sw + lever * Mpsi_sw)

    M1_eq = 0.25 * (Mx_eq - My_eq - lever * M_psi_eq)
    M2_eq = 0.25 * (Mx_eq + My_eq + lever * M_psi_eq)
    M3_eq = 0.25 * (Mx_eq + My_eq - lever * M_psi_eq)
    M4_eq = 0.25 * (Mx_eq - My_eq + lever * M_psi_eq)

    Mi_sw = SVector(M1_sw, M2_sw, M3_sw, M4_sw)
    Mi_eq = SVector(M1_eq, M2_eq, M3_eq, M4_eq)

    # Smooth adaptive-gain dynamics (identical structure; driven by the new s)
    base_dK_x   = asmc.gamma_x   * (s_x   * ss_x)
    base_dK_y   = asmc.gamma_y   * (s_y   * ss_y)
    base_dK_psi = asmc.gamma_psi * (s_psi * ss_psi)
    dK_x   = base_dK_x   * smooth_bound(K_x,   asmc.K_max_x)   - 0.1 * (K_x   / asmc.K_max_x)^3 - asmc.sigma_x   * (K_x   - asmc.K_x0   * 0.95) * exp(asmc.decay_k*(1 - s_x^2  /(9 * asmc.eps_psi^2)))
    dK_y   = base_dK_y   * smooth_bound(K_y,   asmc.K_max_y)   - 0.3 * (K_y   / asmc.K_max_y)^3 - asmc.sigma_y   * (K_y   - asmc.K_y0   * 0.95) * exp(asmc.decay_k*(1 - s_y^2  /(9 * asmc.eps_psi^2)))
    dK_psi = base_dK_psi * smooth_bound(K_psi, asmc.K_max_psi) - 0.5 * (K_psi / asmc.K_max_psi)^3 - asmc.sigma_psi * (K_psi - asmc.K_psi0 * 0.95) * exp(asmc.decay_k*(1 - s_psi^2/(9 * asmc.eps_psi^2)))

    return Mi_sw, Mi_eq, SVector(dK_x, dK_y, dK_psi), delta_hat
end

### System ODE: `dynamics_full_mf_asmc!`

In-place RHS for `OrdinaryDiffEq`. For each timestep it:

1. Unpacks the 39-D state.
2. Computes roller contact geometry (`φ̃ᵢ`, `ΔYᵢ`) through the smoothed sawtooth.
3. Computes contact-point velocities `Vpᵢ` and spin velocity `ω_{zi}` from Eq. (6) of the paper.
4. Evaluates the multicomponent friction per wheel to get `(Fxᵢ, Fyᵢ, Mzᵢ)`.
5. Calls `asmc_torques` or `asmc_torques_vel` and applies torque saturation.
6. Assembles platform dynamics `M⁻¹·(∑F + Coriolis)`.
7. Computes wheel dynamics `ω̇ᵢ` and roller dynamics `γ̇ᵢ`.
8. Computes dynamics of adaptive gains, LuGre bristles, and (if switched on) disturbance observers
9. Writes the full 39-D derivative into `du`.

The RHS is written to avoid heap allocations: all 4-vectors are `SVector{4}` and arithmetic is
done scalar-wise inside `ntuple(...)` loops. This is the single biggest Julia perf win over Python.


In [ ]:
# ---------------------------
# In-place dynamics RHS for ODEProblem  (39-D: 21 base + 12 LuGre bristles + 6 observer states)
# p = (params, asmc, chi, p1, p2, coupling, lugre, eso, ctrl)   # ← ctrl added (9th element)
#   ctrl = asmc_torques  (position tracking)  |  asmc_torques_vel  (velocity tracking)
# state: [1:21] base; [22:25]=zx, [26:29]=zy, [30:33]=zs;
#        observer [34:36]=(ζ_x,ζ_y,ζ_ψ) if :linear, else (v̂_x,v̂_y,v̂_ψ); [37:39]=δ̂ if :super_twisting
# ---------------------------
function dynamics_full_mf_asmc!(du, u, p, t)
    params, asmc, chi, p1, p2, coupling, lugre, eso, ctrl = p   # ← unpack ctrl

    Vx, Vy, psi_dot, psi = u[1], u[2], u[3], u[4]
    ti = SVector(u[5],  u[6],  u[7],  u[8])
    wi = SVector(u[9],  u[10], u[11], u[12])
    gi = SVector(u[13], u[14], u[15], u[16])
    zx = SVector(u[22], u[23], u[24], u[25])
    zy = SVector(u[26], u[27], u[28], u[29])
    zs = SVector(u[30], u[31], u[32], u[33])

    px, py = params.wc_x, params.wc_y
    R, Rd  = params.R, params.Ra
    delta  = params.delta
    aX, aY = params.aX, params.aY
    ms, m  = params.ms, params.m

    sdi = sin.(delta); cdi = cos.(delta); tdi = tan.(delta)
    ti_t  = sawtooth_approx.(ti)
    sti_t = sin.(ti_t);  cti_t = cos.(ti_t);  tti_t = tan.(ti_t)
    DYi   = Rd .* tdi .* tti_t

    Vpi_x = @. Vx - psi_dot * (py + DYi) - wi * R +
               gi * sdi * (Rd * cti_t - R) + DYi * gi * cdi * sti_t
    Vpi_y = @. Vy + psi_dot * px +
               gi * cdi * (R * cti_t - Rd)
    wzi   = @. psi_dot - gi * (-sti_t * cdi)

    Ni = params.N_per_roller

    # Per-wheel DYNAMIC LuGre: forces + bristle derivatives.
    fr = ntuple(4) do i
        lugre_dyn_rates(lugre, coupling, params.f_coulomb, Ni[i], chi,
                        wzi[i], Vpi_x[i], Vpi_y[i], zx[i], zy[i], zs[i])
    end
    Fx_i = SVector(fr[1][1], fr[2][1], fr[3][1], fr[4][1])
    Fy_i = SVector(fr[1][2], fr[2][2], fr[3][2], fr[4][2])
    Mz_i = SVector(fr[1][3], fr[2][3], fr[3][3], fr[4][3])
    dzx  = SVector(fr[1][4], fr[2][4], fr[3][4], fr[4][4])
    dzy  = SVector(fr[1][5], fr[2][5], fr[3][5], fr[4][5])
    dzs  = SVector(fr[1][6], fr[2][6], fr[3][6], fr[4][6])

    Mi_sw, Mi_eq, dK, _ = ctrl(u, t, params, asmc, eso)   # ← injected controller
    Mi_total = Mi_sw .+ Mi_eq
    max_tau  = params.Max_torque
    Mi_sat   = smooth_sat.(Mi_total, max_tau, 4)

    # ---- MIMO DOB auxiliary-state ODE  (derivation eq 12) ----
    l, h  = params.l, params.h
    lever = R / (l + h)
    M_x_applied   =  Mi_sat[1] + Mi_sat[2] + Mi_sat[3] + Mi_sat[4]
    M_y_applied   = -Mi_sat[1] + Mi_sat[2] + Mi_sat[3] - Mi_sat[4]
    M_psi_applied = (-Mi_sat[1] + Mi_sat[2] - Mi_sat[3] + Mi_sat[4]) / lever
    F_phys = SVector(M_x_applied / R, M_y_applied / R, M_psi_applied)
    v_body = SVector(Vx, Vy, psi_dot)
    en     = eso.enable ? 1.0 : 0.0
    a_nom  = params.M_aug_inv * F_phys

    Mz_rolleri = @. px * Fy_i - py * Fx_i + Mz_i
    RHS0 = sum(Fx_i)          + ms * psi_dot * Vy  + m * aX * psi_dot^2
    RHS1 = sum(Fy_i)          - ms * psi_dot * Vx  + m * aY * psi_dot^2
    RHS2 = sum(Mz_rolleri)    - m * psi_dot * (aX * Vx + aY * Vy)
    dv = params.M_inv * SVector(RHS0, RHS1, RHS2)

    dwi = @. (Mi_sat - Fx_i * R - p1 * wi) / params.J_wheel
    dgi = @. (- p2 * gi
              - Fx_i * (sdi * (R - Rd * cti_t) - DYi * sti_t * cdi)
              - Fy_i * cdi * (Rd - R * cti_t)
              - Mz_i * sti_t * cdi) / params.J_roller / params.rollers_per_wheel

    du[1] = dv[1];  du[2] = dv[2];  du[3] = dv[3]
    du[4] = psi_dot
    @inbounds for i in 1:4
        du[4+i]  = wi[i]
        du[8+i]  = dwi[i]
        du[12+i] = dgi[i]
    end
    du[17] = dK[1]; du[18] = dK[2]; du[19] = dK[3]
    du[20] = Vx * cos(psi) - Vy * sin(psi)
    du[21] = Vx * sin(psi) + Vy * cos(psi)
    @inbounds for i in 1:4
        du[21+i] = dzx[i]   # u[22:25]
        du[25+i] = dzy[i]   # u[26:29]
        du[29+i] = dzs[i]   # u[30:33]
    end
    if eso.kind == :super_twisting
        v_hat = SVector(u[34], u[35], u[36])
        d_hat = SVector(u[37], u[38], u[39])
        e_obs = v_body - v_hat
        k1    = SVector(eso.k1_x, eso.k1_y, eso.k1_psi)
        k2    = SVector(eso.k2_x, eso.k2_y, eso.k2_psi)
        phi1  = reg_sqrt.(e_obs, eso.eps_obs)
        phi2  = reg_sign.(e_obs, eso.eps_obs)
        dvhat = en * (a_nom + d_hat + k1 .* phi1)
        ddhat = en * (k2 .* phi2)
        du[34] = dvhat[1]; du[35] = dvhat[2]; du[36] = dvhat[3]
        du[37] = ddhat[1]; du[38] = ddhat[2]; du[39] = ddhat[3]
    else
        zeta_vec = SVector(u[34], u[35], u[36])
        Omega    = SVector(eso.omega_o_x, eso.omega_o_y, eso.omega_o_psi)
        dzeta    = en * (-Omega .* zeta_vec - Omega.^2 .* v_body - Omega .* a_nom)
        du[34] = dzeta[1]; du[35] = dzeta[2]; du[36] = dzeta[3]
        du[37] = 0.0;      du[38] = 0.0;      du[39] = 0.0
    end
    return nothing
end

### Roller quasi-staticity, dwell time & ASMC torque-split diagnostics

In [ ]:
# ---------------------------
# DIAGNOSTICS (post-processing on `sol`) — three quantities, no change to the RHS:
#   (1) t_dwell_i = (2π / N_rollers) / |ωᵢ|   — time one roller stays in contact.
#   (2) Roller-spin torque balance.  The spin ODE is
#         J_eff·γ̈ᵢ = Tdamp + Tfx + Tfy + Tmz ,   J_eff = J_roller · N_rollers.
#       Tnet := J_eff·γ̈ᵢ is the inertial torque (= net of the four).  The spin is
#       slaved (algebraic) where  resid = |Tnet| / (|Tdamp|+|Tfx|+|Tfy|+|Tmz|) ≪ 1,
#       i.e. the contact/damping terms cancel and inertia carries a negligible residual.
#       Single-roller inertial torque is Tnet / N_rollers (J_roller·γ̈) — the literal
#       "J_roller·γ̇-rate" — reported via Jeff so you can divide if you want it.
#   (3) Per-wheel control split:
#         M_eff = no-slip inverse-dynamics equivalent control,
#         M_dob = MIMO-DOB compensation  (recovered from δ̂:  ΔW = -diag(R,R,1)·M_aug·δ̂,
#                 mapped through the same 0.25 O-config allocation),
#         M_sw  = sliding switching wrench,   M_sat = applied (saturated) torque.
#       M_eff = Mᵢ_eq − M_dob.  Small M_dob/M_sw relative to M_eff ⇒ the no-slip
#       equivalent control already does the work, i.e. the slip model adds little here.
# `coupling/lugre/eso/ctrl/p1/p2` must match the run that produced `sol`.
# ---------------------------
function compute_roller_ctrl_diag(sol, params::PlatformParams, asmc::ASMCParams,
                                  eso::ESOParams, chi::Real, coupling::Symbol,
                                  lugre::LuGreParams, ctrl::Function, p1::Real, p2::Real)
    N  = length(sol.t)
    Nr = params.rollers_per_wheel
    Jeff = params.J_roller * Nr

    tdwell = zeros(4, N)
    Tdamp  = zeros(4, N); Tfx = zeros(4, N); Tfy = zeros(4, N); Tmz = zeros(4, N)
    Tnet   = zeros(4, N); resid = zeros(4, N)
    Meff   = zeros(4, N); Mdob = zeros(4, N); Msw = zeros(4, N); Msat = zeros(4, N)

    sdi = sin.(params.delta); cdi = cos.(params.delta); tdi = tan.(params.delta)
    px, py = params.wc_x, params.wc_y
    R, Rd  = params.R, params.Ra
    l, h   = params.l, params.h
    lever  = R / (l + h)
    w_floor = 1e-4    # avoid Inf when a wheel is ~stationary (no roller handoff there)

    @inbounds for k in 1:N
        t = sol.t[k]; u = sol.u[k]
        Vx, Vy, psi_dot, psi = u[1], u[2], u[3], u[4]
        ti = SVector(u[5],  u[6],  u[7],  u[8])
        wi = SVector(u[9],  u[10], u[11], u[12])
        gi = SVector(u[13], u[14], u[15], u[16])
        zx = SVector(u[22], u[23], u[24], u[25])
        zy = SVector(u[26], u[27], u[28], u[29])
        zs = SVector(u[30], u[31], u[32], u[33])

        ti_t  = sawtooth_approx.(ti)
        sti_t = sin.(ti_t); cti_t = cos.(ti_t); tti_t = tan.(ti_t)
        DYi   = Rd .* tdi .* tti_t

        Vpi_x = @. Vx - psi_dot * (py + DYi) - wi * R +
                   gi * sdi * (Rd * cti_t - R) + DYi * gi * cdi * sti_t
        Vpi_y = @. Vy + psi_dot * px + gi * cdi * (R * cti_t - Rd)
        wzi   = @. psi_dot - gi * (-sti_t * cdi)

        for i in 1:4
            fx, fy, mz, _, _, _ = lugre_dyn_rates(lugre, coupling, params.f_coulomb,
                                                  params.N_per_roller[i], chi,
                                                  wzi[i], Vpi_x[i], Vpi_y[i], zx[i], zy[i], zs[i])
            # roller-spin torque contributions (signs as they enter J_eff·γ̈ = Σ)
            A = sdi[i] * (R - Rd * cti_t[i]) - DYi[i] * sti_t[i] * cdi[i]
            B = cdi[i] * (Rd - R * cti_t[i])
            C = sti_t[i] * cdi[i]
            td = -p2 * gi[i]; tx = -fx * A; ty = -fy * B; tm = -mz * C
            net = td + tx + ty + tm                       # = J_eff·γ̈ᵢ
            Tdamp[i,k]=td; Tfx[i,k]=tx; Tfy[i,k]=ty; Tmz[i,k]=tm; Tnet[i,k]=net
            resid[i,k] = abs(net) / (abs(td)+abs(tx)+abs(ty)+abs(tm) + eps())
            tdwell[i,k] = (2π / Nr) / max(abs(wi[i]), w_floor)
        end

        Mi_sw, Mi_eq, _, dhat = ctrl(u, t, params, asmc, eso)
        Mdh = params.M_aug * dhat                          # SVector{3}
        dWx = -R * Mdh[1]; dWy = -R * Mdh[2]; dWp = -Mdh[3]
        md = SVector(0.25*(dWx - dWy - lever*dWp),
                     0.25*(dWx + dWy + lever*dWp),
                     0.25*(dWx + dWy - lever*dWp),
                     0.25*(dWx - dWy + lever*dWp))
        for i in 1:4
            Mdob[i,k] = md[i]
            Meff[i,k] = Mi_eq[i] - md[i]                   # no-slip inverse-dynamics part
            Msw[i,k]  = Mi_sw[i]
            Msat[i,k] = params.Max_torque * tanh((Mi_sw[i] + Mi_eq[i]) / params.Max_torque)
        end
    end
    return (; tdwell, Tdamp, Tfx, Tfy, Tmz, Tnet, resid, Meff, Mdob, Msw, Msat, Jeff, Nr)
end


### Single-run driver

Wraps one `solve()` call: builds the `ODEProblem` with an autodiff Jacobian, picks a stiff
solver, runs with a progress bar, and returns `sol`.

**Why `Rodas5P` as the default?** It's an L-stable Rosenbrock-W method — stiffly stable, 5th
order, with embedded error estimate for adaptive step control. It handles the ~10⁴ timescale
ratio between platform and roller dynamics cleanly at `rtol=1e-6` without step rejection storms.
Alternatives worth benchmarking: `QNDF` (BDF, good for long runs), `TRBDF2` (cheaper per step,
2nd order), `KenCarp47` (ESDIRK, often fastest for this problem class).


In [ ]:

# ---------------------------
# Initial-condition construction (39-D state)
# ---------------------------
function build_initial_state(params::PlatformParams, asmc::ASMCParams)
    u0 = zeros(39)        # 21 base + 12 LuGre bristles + 6 observer states
    ref = Profiles.active_ref()
    u0[1] = 0.0            # Vx     — all profiles start from rest by design
    u0[2] = 0.0            # Vy
    u0[3] = 0.0            # psi_dot (profiles guarantee Wz(0)=0)
    u0[4] = ref.psi(0.0)   # ψ₀ = reference heading at t=0 ⇒ no initial yaw transient
    u0[5:8]  .= 0.1        # θᵢ
    u0[9:12] .= 0.0        # ωᵢ
    u0[13:16].= 0.0        # γᵢ
    u0[17] = 1.2 * asmc.K_x0
    u0[18] = 1.2 * asmc.K_y0
    u0[19] = 1.2 * asmc.K_psi0
    # u0[20:21] (world position Xo, Yo) stay 0 — VelRef profiles carry no position
    # reference, and PosRef profiles (ellipse) start their path AT the origin.
    # carry no position reference; the world path is whatever ∫R(ψ)v dt gives.
    return u0
end


# Per-group absolute tolerances — UNCHANGED from the previous cell
const _AT = (
    body_vel = 1.0e-8,   psi      = 1.0e-7,
    wtheta   = 1.0e-8,   womega   = 1.0e-7,
    gamma    = 1.0e-7,   gains    = 1.0e-7,
    worldpos = 1.0e-7,   bristle  = 1.0e-10,
    bristle_rot = 1.0e-7, observer = 1.0e-7,  dist_est = 1.0e-7,
)
const ABSTOL = vcat(
    fill(_AT.body_vel, 3), fill(_AT.psi, 1),
    fill(_AT.wtheta, 4),   fill(_AT.womega, 4),
    fill(_AT.gamma, 4),    fill(_AT.gains, 3),
    fill(_AT.worldpos, 2), fill(_AT.bristle, 8),
    fill(_AT.bristle_rot, 4), fill(_AT.observer, 3), fill(_AT.dist_est, 3),
)   # 3+1+4+4+4+3+2+8+4+3+3 = 39
  # counts: 3+1+4+4+4+3+2+12+3+3 = 39

# ---------------------------
# Single-run driver — T and tstops come from the active reference (either kind)
# ---------------------------
function run_one_chi(chi::Real, params::PlatformParams, asmc::ASMCParams;
                     friction_model::Symbol = friction_model, lugre::LuGreParams = lugre,
                     eso::ESOParams = ESOParams(),
                     T::Real = Profiles.active_ref().T_total,
                     friction_case::Int = friction_case,
                     dt_save::Real = 5e-4,                 # 2 kHz master resolution
                     reltol::Real = 1e-8, abstol::Union{Real,AbstractVector} = ABSTOL,
                     solver = TRBDF2())
    p1, p2 = friction_case == 1 ? (params.p1_case1, params.p2_case1) :
                                   (params.p1_case2, params.p2_case2)
    u0 = build_initial_state(params, asmc)
    t_eval = collect(range(0.0, T; length = round(Int, T / dt_save) + 1))
    ctrl = is_velref() ? asmc_torques_vel : asmc_torques   # reference kind decides
    p = (params, asmc, chi, p1, p2, coupling_of(friction_model), lugre, eso, ctrl)

    prob = ODEProblem(dynamics_full_mf_asmc!, u0, (0.0, T), p)

    prog = ProgressMeter.Progress(100; desc=@sprintf("%-16s chi=%.4f ", String(friction_model), chi))
    last_pct = Ref(0)
    function update_prog!(integrator)
        pct = clamp(floor(Int, 100 * integrator.t / T), 0, 100)
        if pct > last_pct[]
            ProgressMeter.update!(prog, pct); last_pct[] = pct
        end
        return nothing
    end
    pbar_cb = PeriodicCallback(update_prog!, T/100; initial_affect = false)

    sol = solve(prob, solver;
                reltol = reltol, abstol = abstol, saveat = t_eval,
                tstops = Profiles.active_ref().tstops,        # profile phase boundaries
                callback = pbar_cb, dtmax = 0.001, maxiters = 10^7)
    ProgressMeter.finish!(prog)
    return sol
end



### Main simulation loop over friction model × `χ`

- `friction_models` = both models when `compare_models = true`, else just `friction_model`.
- `χ` is swept over the **full** training ∪ held-out set so the comparison is meaningful in every run.
- `write_data = true`  → dump Arrow+JLD2(+CSV) per (model, χ), with the model in the filename **and**
  a `friction_model` column; plots are split into **Training χ** and **Held-out χ** sections.
- `write_data = false` → no files; all χ plotted together (ungrouped).

(To iterate quickly, trim `train_chi_values` / `heldout_chi_values` in the parameters cell.)


In [ ]:
# ---------------------------
# Main loop — friction model × χ for the trajectory built in Cell 14.
# Plotting always; disk writing only behind write_data (notebook-only gate;
# the sweep driver always writes and never plots).
# ---------------------------
params = PlatformParams(BASE; mu_friction = mu_friction)
asmc   = ASMCParams(eps = max(PEAK_V * 0.025, 0.001))
eso = ESOParams(kind = dob_kind,
                omega_o_x = omega_o_x, omega_o_y = omega_o_y, omega_o_psi = omega_o_psi,
                k1_x = k1_x, k2_x = k2_x, k1_y = k1_y, k2_y = k2_y,
                k1_psi = k1_psi, k2_psi = k2_psi, eps_obs = eps_obs,
                enable = use_dob)

friction_models = compare_models ? [:lugre_uncoupled, :lugre_adamov] : [friction_model]
chi_values      = sort(unique(vcat(train_chi_values, heldout_chi_values)))
ctrl            = is_velref() ? asmc_torques_vel : asmc_torques
write_data && (isdir(output_dir) || mkpath(output_dir))

all_labels = Dict{Tuple{Symbol,Float64},Any}()
all_sols   = Dict{Tuple{Symbol,Float64},Any}()
all_paths  = Dict{Tuple{Symbol,Float64},Any}()

for fm in friction_models, chi in chi_values
    tag = chi in heldout_chi_values ? "HELD-OUT" : "train"
    @info "Running" profile=PROFILE_NAME model=fm chi=chi split=tag
    sol    = run_one_chi(chi, params, asmc; friction_model=fm, lugre=lugre, eso=eso)
    labels = DataStore.compute_labels(sol, params, asmc, chi, coupling_of(fm), lugre, eso;
                                      friction_fn   = lugre_dyn_rates,
                                      controller_fn = ctrl,
                                      sawtooth_fn   = sawtooth_approx)
    meta = (profile = PROFILE_NAME,
            combo_idx = Int(get(TRAJ_CFG, "combo_idx", 0)),
            mu = mu_friction, chi = chi,
            friction_case = friction_case,
            friction_model = fm,
            sweep_seed = PICK_SEED)
    df = DataStore.assemble_dataframe(sol, labels, REF, meta)
    if write_data
        paths = DataStore.write_outputs(df, sol, labels, params, asmc, meta;
                                        outdir = output_dir, cfg = TRAJ_CFG)
        all_paths[(fm, chi)] = paths
        @info "  wrote" paths.arrow
    end
    all_sols[(fm, chi)]   = sol
    all_labels[(fm, chi)] = labels
end
println("Done. (model, χ) runs: ", sort(collect(keys(all_sols))))

### Plots

Mirrors the Python v4 `plot_results`: velocities, heading, wheel angular velocities,
trajectory, and control torques — one panel per `χ` on the same figure for comparison.


In [ ]:
# ---------------------------
# Comprehensive per-χ diagnostics, comparing the two friction models.
# Both models are overlaid in every panel: uncoupled = dashed/blue, adamov = solid/crimson.
# ---------------------------
const MODEL_STYLE = Dict(:lugre_uncoupled => :dash,      :lugre_adamov => :solid)
const MODEL_COLOR = Dict(:lugre_uncoupled => :steelblue, :lugre_adamov => :crimson)

# per-wheel (4×N) data matrix for a given quantity
function wheel_matrix(fm, chi, kind)
    sol = all_sols[(fm, chi)]; lab = all_labels[(fm, chi)]
    U = hcat(sol.u...)
    kind === :omega      && return U[9:12,  :]    # wheel angular velocity ωᵢ
    kind === :rollerspin && return U[13:16, :]    # roller spin γ̇ᵢ
    kind === :torque     && return lab.Msat       # applied (saturated) control torque
    kind === :Fx         && return lab.Fxs
    kind === :Fy         && return lab.Fys
    kind === :Mz         && return lab.Mzs
    error("unknown kind $kind")
end

# Desired world path: integrate R(ψ_des)·[Vx_d, Vy_d] (trapezoid). VelRef has
# no position channel — this reconstructs the implied reference path, which
# starts at (0,0) exactly like the platform IC.
function ref_world_path(ref, tt)
    Vxd = [ref.Vx(t) for t in tt];  Vyd = [ref.Vy(t) for t in tt]
    ps  = [ref.psi(t) for t in tt]
    Vxw = @. Vxd * cos(ps) - Vyd * sin(ps)
    Vyw = @. Vxd * sin(ps) + Vyd * cos(ps)
    X = zeros(length(tt)); Y = zeros(length(tt))
    for k in 2:length(tt)
        dt = tt[k] - tt[k-1]
        X[k] = X[k-1] + 0.5 * (Vxw[k] + Vxw[k-1]) * dt
        Y[k] = Y[k-1] + 0.5 * (Vyw[k] + Vyw[k-1]) * dt
    end
    return X, Y
end

# (1) Tracking: Vx, Vy, ψ̇, path — desired vs actual, for ONE model.
# References are body-frame natively now — no global_to_local_frame needed.
function plot_tracking(chi, fm)
    sol = all_sols[(fm, chi)]; U = hcat(sol.u...); tt = sol.t
    nm = String(fm); c = MODEL_COLOR[fm]
    if hasproperty(REF, :Vx)                      # VelRef — body-frame native
        Vxd  = [REF.Vx(t) for t in tt]
        Vyd  = [REF.Vy(t) for t in tt]
        wdes = [REF.Wz(t) for t in tt]
    else                                          # PosRef (ellipse)
        des  = [global_to_local_frame(tt[k], U[4,k], REF.Vxo, REF.Vyo, REF.om)
                for k in eachindex(tt)]
        Vxd  = [d[1] for d in des];  Vyd = [d[2] for d in des]
        wdes = [REF.om(t) for t in tt]
    end

    pVx = plot(title="Vx  des vs act", legend=:best)
    plot!(pVx, tt, Vxd;    label="desired", c=:black, ls=:dot, lw=2)
    plot!(pVx, tt, U[1,:]; label="actual",  c=c, lw=2)
    pVy = plot(title="Vy  des vs act")
    plot!(pVy, tt, Vyd;    label="desired", c=:black, ls=:dot, lw=2)
    plot!(pVy, tt, U[2,:]; label="actual",  c=c, lw=2)
    pW  = plot(title="ψ̇  des vs act")
    plot!(pW,  tt, wdes;   label="desired", c=:black, ls=:dot, lw=2)
    plot!(pW,  tt, U[3,:]; label="actual",  c=c, lw=2)
    Xr, Yr = hasproperty(REF, :Vx) ? ref_world_path(REF, tt) :
                 ([REF.xo(t) for t in tt], [REF.yo(t) for t in tt])
    pXY = plot(title="Path  implied-ref vs act", aspect_ratio=1)
    plot!(pXY, Xr, Yr;           label="implied reference", c=:black, ls=:dot, lw=2)
    plot!(pXY, U[20,:], U[21,:]; label="actual",            c=c, lw=2)

    plot(pVx, pVy, pW, pXY; layout=(2,2), size=(1200,900),
         plot_title="Tracking — $nm (χ=$chi)")
end

# 2×2 per-wheel comparison (all 4 wheels side by side, both models overlaid)
function plot_4wheel(chi, models, kind, qtitle)
    plts = []
    for i in 1:4
        p = plot(title="$qtitle — wheel $i", legend = (i==1 ? :best : false))
        for fm in models
            tt = all_sols[(fm, chi)].t
            plot!(p, tt, wheel_matrix(fm, chi, kind)[i, :];
                  label=String(fm), ls=MODEL_STYLE[fm], c=MODEL_COLOR[fm], lw=2)
        end
        push!(plts, p)
    end
    plot(plts...; layout=(2,2), size=(1200,900), plot_title="$qtitle (χ=$chi)")
end

# (3a) ASMC gains Kx, Ky, Kψ side by side, both models
function plot_gains(chi, models)
    names = ["Kx","Ky","Kψ"]; idx = [17,18,19]; plts = []
    for (j, (lbl, id)) in enumerate(zip(names, idx))
        p = plot(title=lbl, legend = (j==1 ? :best : false))
        for fm in models
            tt = all_sols[(fm,chi)].t; U = hcat(all_sols[(fm,chi)].u...)
            plot!(p, tt, U[id, :]; label=String(fm),
                  ls=MODEL_STYLE[fm], c=MODEL_COLOR[fm], lw=2)
        end
        push!(plts, p)
    end
    plot(plts...; layout=(1,3), size=(1500,450), plot_title="ASMC gains (χ=$chi)")
end

# (3b) Bristle states: rows zx, zy, z_rot ; columns wheels 1..4 (both models)
function plot_bristles(chi, models)
    rows = [("zx",21), ("zy",25), ("z_rot",29)]   # wheel i at state index off+i
    plts = []
    for (rlbl, off) in rows, i in 1:4
        p = plot(title="$rlbl  wheel $i", legend = (rlbl=="zx" && i==1 ? :best : false))
        for fm in models
            tt = all_sols[(fm,chi)].t; U = hcat(all_sols[(fm,chi)].u...)
            plot!(p, tt, U[off+i, :]; label=String(fm),
                  ls=MODEL_STYLE[fm], c=MODEL_COLOR[fm], lw=2)
        end
        push!(plts, p)
    end
    plot(plts...; layout=(3,4), size=(1800,1100), plot_title="Bristle states (χ=$chi)")
end

# (4) Contact forces resolved on roller axes / spin torque
#
# Each row decomposes the contact friction wrench onto the i-th wheel's
# roller-axis frame:
#   F∥   =   Fxᵢ·cos δᵢ + Fyᵢ·sin δᵢ    — along the roller spin axis eᵢ
#                                          ("free-rolling" direction; small —
#                                          only roller-bearing drag and microslip)
#   F⊥   =  -Fxᵢ·sin δᵢ + Fyᵢ·cos δᵢ    — perpendicular to eᵢ in the contact plane
#                                          ("drive" direction; this is what
#                                          transmits wheel torque to the platform)
#   Mz                                    — spin friction torque about ẑ
#                                          (invariant under in-plane rotation,
#                                          shown unprojected)
#
# δᵢ = ∠(X̂_body, eᵢ) from PlatformParams.delta (O-config: -45°,+45°,+45°,-45°).
# Both Fxᵢ,Fyᵢ and δᵢ live in the platform body frame, so the projection is direct.
function plot_forces(chi, models)
    delta = params.delta

    rows = [(:Fpar,  "F∥  (along eᵢ)"),
            (:Fperp, "F⊥  (perp to eᵢ)"),
            (:Mz,    "Mz")]
    plts = []
    for (kind, rlbl) in rows, i in 1:4
        cδ, sδ = cos(delta[i]), sin(delta[i])
        p = plot(title = "$rlbl  wheel $i",
                 legend = (kind === :Fpar && i == 1 ? :best : false))
        for fm in models
            tt = all_sols[(fm, chi)].t
            series = if kind === :Mz
                wheel_matrix(fm, chi, :Mz)[i, :]
            else
                Fx = wheel_matrix(fm, chi, :Fx)[i, :]
                Fy = wheel_matrix(fm, chi, :Fy)[i, :]
                kind === :Fpar ?  Fx .* cδ .+ Fy .* sδ :
                                 -Fx .* sδ .+ Fy .* cδ
            end
            plot!(p, tt, series; label = String(fm),
                  ls = MODEL_STYLE[fm], c = MODEL_COLOR[fm], lw = 2)
        end
        push!(plts, p)
    end
    plot(plts...; layout=(3,4), size=(1800,1100),
         plot_title="Contact forces on roller axes / spin torque (χ=$chi)")
end


# (5) Yaw disturbance-observer comparison across friction models (both overlaid).
#   ψ̇ and ψ tracking, the DOB estimate δ̂_ψ, and the adaptive yaw gain K_ψ
#   (which should now settle low, since the DOB carries the yaw disturbance).
function plot_dob_comparison(chi, models)
    pW   = plot(title="ψ̇  des vs act", legend=:best)
    pPsi = plot(title="ψ  des vs act", legend=false)
    pD   = plot(title="δ̂_ψ  (MIMO-DOB estimate)", legend=false)
    pK   = plot(title="K_ψ  (adaptive yaw gain)", legend=false)
    for fm in models
        tt  = all_sols[(fm, chi)].t
        U   = hcat(all_sols[(fm, chi)].u...)
        lab = all_labels[(fm, chi)]
        ls  = MODEL_STYLE[fm]; c = MODEL_COLOR[fm]
        plot!(pW,   tt, U[3, :];  label=String(fm), ls=ls, c=c, lw=2)
        plot!(pPsi, tt, U[4, :];  label=String(fm), ls=ls, c=c, lw=2)
        plot!(pD,   tt, lab.Dhat_psi; label=String(fm), ls=ls, c=c, lw=2)
        plot!(pK,   tt, U[19, :]; label=String(fm), ls=ls, c=c, lw=2)
    end
    tt0 = all_sols[(first(models), chi)].t
    wdes0 = hasproperty(REF, :Wz) ? [REF.Wz(t) for t in tt0] : [REF.om(t) for t in tt0]
    plot!(pW,   tt0, wdes0; label="desired", c=:black, ls=:dot, lw=2)
    plot!(pPsi, tt0, [REF.psi(t) for t in tt0]; label="desired", c=:black, ls=:dot, lw=2)
    plot(pW, pPsi, pD, pK; layout=(2,2), size=(1200,900),
         plot_title="Yaw DOB comparison (χ=$chi)")
end

# (5b) Lateral (Y) disturbance-observer comparison across friction models (both overlaid).
#   Vy tracking, the DOB estimate δ̂_y, and the adaptive y gain K_y
#   (which should now settle low, since the DOB carries the lateral disturbance).
function plot_dob_comparison_y(chi, models)
    pVy = plot(title="Vy  des vs act", legend=:best)
    pD  = plot(title="δ̂_y  (MIMO-DOB estimate)", legend=false)
    pK  = plot(title="K_y  (adaptive y gain)", legend=false)
    for fm in models
        tt  = all_sols[(fm, chi)].t
        U   = hcat(all_sols[(fm, chi)].u...)
        lab = all_labels[(fm, chi)]
        ls  = MODEL_STYLE[fm]; c = MODEL_COLOR[fm]
        plot!(pVy, tt, U[2, :];    label=String(fm), ls=ls, c=c, lw=2)
        plot!(pD,  tt, lab.Dhat_y; label=String(fm), ls=ls, c=c, lw=2)
        plot!(pK,  tt, U[18, :];   label=String(fm), ls=ls, c=c, lw=2)
    end
    tt0 = all_sols[(first(models), chi)].t
    U0  = hcat(all_sols[(first(models), chi)].u...)
    Vyd0 = hasproperty(REF, :Vy) ? [REF.Vy(t) for t in tt0] :
           [global_to_local_frame(tt0[k], U0[4, k], REF.Vxo, REF.Vyo, REF.om)[2]
            for k in eachindex(tt0)]
    plot!(pVy, tt0, Vyd0; label="desired", c=:black, ls=:dot, lw=2)
    plot(pVy, pD, pK; layout=(1,3), size=(1500,450),
         plot_title="Lateral (Y) DOB comparison (χ=$chi)")
end


# (5c) Lateral (X) disturbance-observer comparison across friction models (both overlaid).
#   Vx tracking, the DOB estimate δ̂_x, and the adaptive x gain K_x.
function plot_dob_comparison_x(chi, models)
    pVx = plot(title="Vx  des vs act",       legend=:best)
    pD  = plot(title="δ̂_x  (MIMO-DOB est.)", legend=false)
    pK  = plot(title="K_x  (adaptive X gain)", legend=false)
    for fm in models
        tt  = all_sols[(fm, chi)].t
        U   = hcat(all_sols[(fm, chi)].u...)
        lab = all_labels[(fm, chi)]
        ls  = MODEL_STYLE[fm]; c = MODEL_COLOR[fm]
        plot!(pVx, tt, U[1, :];    label=String(fm), ls=ls, c=c, lw=2)
        plot!(pD,  tt, lab.Dhat_x; label=String(fm), ls=ls, c=c, lw=2)
        plot!(pK,  tt, U[17, :];   label=String(fm), ls=ls, c=c, lw=2)
    end
    tt0 = all_sols[(first(models), chi)].t
    U0  = hcat(all_sols[(first(models), chi)].u...)
    Vxd0 = hasproperty(REF, :Vx) ? [REF.Vx(t) for t in tt0] :
           [global_to_local_frame(tt0[k], U0[4, k], REF.Vxo, REF.Vyo, REF.om)[1]
            for k in eachindex(tt0)]
    plot!(pVx, tt0, Vxd0; label="desired", c=:black, ls=:dot, lw=2)
    plot(pVx, pD, pK; layout=(1,3), size=(1500,450),
         plot_title="Lateral (X) DOB comparison (χ=$chi)")
end

# Full diagnostic set for one χ, in the requested order
function render_all(chi, models)
    for fm in models                                                       # (1) tracking
        display(plot_tracking(chi, fm))                                    #     one model after the other
    end
    display(plot_4wheel(chi, models, :omega,      "Wheel angular velocity ω"))   # (2)
    display(plot_4wheel(chi, models, :torque,     "Control torque (applied)"))   # (2)
    display(plot_gains(chi, models))                                             # (3a)
    display(plot_bristles(chi, models))                                          # (3b)
    display(plot_4wheel(chi, models, :rollerspin, "Roller spin γ̇"))             # (4)
    display(plot_forces(chi, models))                                            # (4)
    if use_dob
        display(plot_dob_comparison_x(chi, models))                                  # (5a) X DOB
        display(plot_dob_comparison_y(chi, models))                                  # (5b) Y DOB
        display(plot_dob_comparison(chi, models))                                   # (5c) yaw DOB
    end
end

if write_data
    println("================  TRAINING χ  ================")
    for chi in sort(train_chi_values);   render_all(chi, friction_models);  end
    println("================  HELD-OUT χ  ================")
    for chi in sort(heldout_chi_values); render_all(chi, friction_models);  end
else
    for chi in chi_values
        render_all(chi, friction_models)
    end
end

### Plots: roller dwell, roller-spin balance/residual, and control split

In [ ]:
# ---------------------------
# Plot the roller / control diagnostics for every (model, χ) that was run, and
# print the scalar summaries that actually answer the questions:
#   • resid (median / p95)            → is the lumped roller spin quasi-static?
#   • ε = τ_eff / min(t_dwell)         → are we inside the slaving envelope?
#   • RMS M_dob, M_sw vs RMS M_eff     → is slip compensation small vs no-slip eq. control?
# ---------------------------
if !write_data    
    _pct(x, q) = (s = sort(vec(x)); s[clamp(ceil(Int, q*length(s)), 1, length(s))])
    _rms(x)    = sqrt(sum(abs2, x) / length(x))
    
    tau_eff_ms = params.J_roller * params.rollers_per_wheel / params.p2_case1 * 1e3  # sliding worst case
    
    for (fm, chi) in sort(collect(keys(all_sols)))
        sol  = all_sols[(fm, chi)]; tt = sol.t
        ctrl = is_velref() ? asmc_torques_vel : asmc_torques
        d    = compute_roller_ctrl_diag(sol, params, asmc, eso, chi, coupling_of(fm),
                                        lugre, ctrl, params.p1_case1, params.p2_case1)
        nm = String(fm)
    
        # ---- (1) dwell time per wheel, with worst-case τ_eff reference ----
        pd = plot(title="roller dwell t_dwell — $nm (χ=$chi)",
                  xlabel="t [s]", ylabel="t_dwell [ms]", legend=:best)
        for i in 1:4
            plot!(pd, tt, clamp.(d.tdwell[i, :] .* 1e3, 0, 200); label="wheel $i", lw=1.5)
        end
        hline!(pd, [tau_eff_ms]; c=:black, ls=:dash, lw=2,
               label="τ_eff (sliding) ≈ $(round(tau_eff_ms, digits=2)) ms")
        display(pd)
    
        # ---- (2) roller-spin torque balance (2×2 wheels): the four RHS terms + Tnet ----
        pbs = []
        for i in 1:4
            p = plot(title="roller-spin torque — wheel $i", xlabel="t [s]", ylabel="N·m",
                     legend = (i == 1 ? :best : false))
            plot!(p, tt, d.Tdamp[i, :]; label="−p2·γ̇",      lw=1.2)
            plot!(p, tt, d.Tfx[i, :];   label="Fx·(lever)",  lw=1.2)
            plot!(p, tt, d.Tfy[i, :];   label="Fy·(lever)",  lw=1.2)
            plot!(p, tt, d.Tmz[i, :];   label="Mz·(lever)",  lw=1.2)
            plot!(p, tt, d.Tnet[i, :];  label="J_eff·γ̈ (net)", c=:black, lw=2.0)
            push!(pbs, p)
        end
        display(plot(pbs...; layout=(2, 2), size=(1200, 900),
                     plot_title="Roller-spin balance — $nm (χ=$chi)"))
    
        # residual ratio (log scale): |inertial| / Σ|terms|
        pr = plot(title="quasi-static residual  |J_eff·γ̈| / Σ|terms| — $nm (χ=$chi)",
                  xlabel="t [s]", ylabel="resid (—)", yscale=:log10, legend=:best)
        for i in 1:4
            plot!(pr, tt, max.(d.resid[i, :], 1e-12); label="wheel $i", lw=1.3)
        end
        hline!(pr, [0.01]; c=:gray, ls=:dot, label="1% slaving bar")
        display(pr)
    
        # ---- (3) control split per wheel: M_eff | M_dob | M_sw | M_sat ----
        pcs = []
        for i in 1:4
            p = plot(title="control torque — wheel $i", xlabel="t [s]", ylabel="N·m",
                     legend = (i == 1 ? :best : false))
            plot!(p, tt, d.Mdob[i, :]; label="M_dob",               lw=1.2)
            plot!(p, tt, d.Msw[i, :];  label="M_sw",                lw=1.2)
            plot!(p, tt, d.Msat[i, :]; label="M_sat (applied)", c=:black, ls=:dash, lw=1.5)
            plot!(p, tt, d.Meff[i, :]; label="M_eff (no-slip eq.)", lw=1.4)
            push!(pcs, p)
        end
        display(plot(pcs...; layout=(2, 2), size=(1200, 900),
                     plot_title="Control split — $nm (χ=$chi)"))
    
        # ---- scalar summary ----
        res_med = _pct(d.resid, 0.50); res_p95 = _pct(d.resid, 0.95)
        eps_max = tau_eff_ms / (minimum(d.tdwell) * 1e3)          # τ_eff / min dwell
        reff = _rms(d.Meff) + eps()
        @info "diag summary" model=fm chi=chi resid_median=round(res_med, sigdigits=3) resid_p95=round(res_p95, sigdigits=3) eps_max=round(eps_max, sigdigits=3) Mdob_over_Meff=round(_rms(d.Mdob)/reff, sigdigits=3) Msw_over_Meff=round(_rms(d.Msw)/reff, sigdigits=3)
    end
end

### Reloading saved data

Quick demonstration of how the Arrow/JLD2 files round-trip. Arrow is the right choice for the
PINN training pipeline (Python-side); JLD2 is what you want for quick Julia-side inspection.


In [ ]:
if write_data && !isempty(all_paths)
    df_arrow, jld = DataStore.reload_run(first(values(all_paths)))
    @info "Arrow columns" names(df_arrow) size(df_arrow)
    @info "JLD2 keys" collect(keys(jld))
    # Python-side equivalent:
    #   import pyarrow.feather as fe;  df = fe.read_feather("path.arrow")
end

### Where the two models diverge, and the LuGre drift question

**Trajectories that maximise the Model 1 vs. Model 2 gap.** The whole difference is the spin coupling
$\tfrac{8}{3\pi}|\omega_z|\chi$ competing with $\lVert V_p\rVert$, so the gap is largest when contact spin and
translation are *both* significant (and $\chi$ is not tiny):

1. **Tight, fast circle (case 5):** the path yaw rate is set by `T_period` (ψ̇ → 2π/`T_period`). Drop
   `T_period` to ~4–5 s with a modest amplitude so $|\omega_z|\chi \gtrsim \lVert V_p\rVert$.
2. **Lemniscate / infinity (case 4):** the gap spikes at the centre crossing where curvature/yaw peak
   while the platform is still translating — a clean, time-localised divergence.
3. **Rotate-in-place while creeping (a new case 6, not yet defined):** high commanded ψ̇ with small V is the
   sharpest discriminator — Model 1 misses the spin-induced loss of translational capacity entirely.
4. Run the discriminating cases at the held-out **χ = 0.008** to amplify; **χ = 0** and **straight-line**
   (case 1, ψ̇≈0) are the null controls where the models should coincide.

**Does the Adamov modification fix LuGre's pulse-load drift? No — it is orthogonal.** Drift is a
*pre-sliding bristle-dynamics* pathology: under small biased oscillatory forcing the deflection state
integrates a net creep because standard LuGre has no bounded elastic-memory cap. The Adamov term modifies
the *gross-sliding force-sharing* (the mean-slip denominator); it does not touch the $\dot z$ pre-sliding
law. With little spin the coupling term ≈ 0, so the $\dot z$ dynamics are identical to the uncoupled
model — same drift. (Note: the bristle $z$ is now integrated, so drift **can** be observed directly — apply a small biased
oscillatory load (e.g. a dithering reference) and watch the accumulated $\int\dot z$. The Adamov/Mindlin terms
still don't cure it: the cure is an elasto-plastic $\alpha(z)$ gate on $\dot z$ / GMS, not the Adamov or Mindlin terms.)


### Notes & further improvements

1. **Tighter tolerances are cheap here.** `reltol=1e-6, abstol=1e-8` is the recommended default —
   at these tolerances `Rodas5P` is still fast because the stiffness is moderate and the RHS is
   native-compiled. The Python v4 notebook runs at `rtol=1e-2` only because Scipy's pure-Python
   Radau implementation makes tighter tolerances expensive.

2. **Why not a symbolic Jacobian?** `ModelingToolkit.jl` can build an analytic sparse Jacobian
   automatically from this RHS. It's a meaningful further speedup (~2×) for very long runs or
   parameter sweeps, but it requires rewriting the RHS in MTK's tracable style (no `SVector`
   allocation, no `MVector` scratch buffers). Left as future work — the `AutoForwardDiff` path
   we use now already gets most of the benefit.

3. **Parallel `χ` sweep.** Each `run_one_chi` call is independent. If you're generating a large
   dataset, wrap the main loop in `Threads.@threads` (or use `Distributed.jl` for true multi-
   process parallelism across cores). For `Threads.@threads` you must make `params`/`asmc` 
   thread-local or ensure your RHS does not mutate any shared state — the current code satisfies
   that (all thread-local).

4. **Arrow vs CSV filesize.** On a 30-s run sampled at 100 Hz (~3000 rows × ~60 columns), Arrow
   with zstd compression lands around 200–400 kB vs 2–4 MB for CSV. That 10× savings is material
   when you're generating thousands of runs for a PINN.

5. **Roller handoff.** This simulation treats the handoff geometrically via the smoothed sawtooth
   — it's *not* an event-triggered switch. That's why Rosenbrock/BDF methods work: the RHS is
   C∞. If you ever need to model impact transients at handoff (discontinuous roller velocity),
   you'd switch to a stiff DAE formulation or use `VCABM` with event detection via `ContinuousCallback`.
